# Notebook 3: PPO for Intraday Trading - More Stable RL

## Welcome to Advanced RL! 🚀

In Notebook 2, you learned DQN. Now let's explore **PPO (Proximal Policy Optimization)** - one of the most popular and stable RL algorithms!

### What you'll learn:
1. **What is PPO?** and why it's better than DQN
2. **Policy-based vs Value-based** RL
3. **Training a PPO agent** for intraday trading
4. **Comparing PPO with DQN**
5. **Understanding stability and performance**

### DQN vs PPO - Key Differences:

| Aspect | DQN | PPO |
|--------|-----|-----|
| Type | Value-based | Policy-based |
| Actions | Discrete only | Discrete or Continuous |
| Stability | Can be unstable | More stable |
| Speed | Faster per step | Slower but smoother |
| Best for | Simple actions | Complex strategies |

Let's dive in! 💪

## Step 1: Import Libraries

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Add TradeMaster to path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(ROOT)

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from mmcv import Config

# TradeMaster imports
from trademaster.utils import replace_cfg_vals, set_seed
from trademaster.nets.builder import build_net
from trademaster.environments.builder import build_environment
from trademaster.datasets.builder import build_dataset
from trademaster.agents.builder import build_agent
from trademaster.optimizers.builder import build_optimizer
from trademaster.losses.builder import build_loss
from trademaster.trainers.builder import build_trainer
from trademaster.transition.builder import build_transition

# Set random seed
set_seed(42)

print("✅ Libraries imported!")
print(f"🔥 PyTorch: {torch.__version__}")
print(f"🎯 CUDA: {torch.cuda.is_available()}")

## Step 2: What is PPO?

### Understanding PPO:

**Proximal Policy Optimization (PPO)** is a policy gradient method that:

1. **Learns a policy directly** (probability distribution over actions)
2. **Updates carefully** (proximal = nearby, doesn't make huge changes)
3. **More stable** than other policy methods
4. **Works with continuous actions** (e.g., "buy 0.5 contracts")

### Why PPO for Trading?
- ✅ **Stability**: Less likely to break during training
- ✅ **Flexibility**: Can handle complex action spaces
- ✅ **Sample efficient**: Learns from experience better
- ✅ **Industry standard**: Used by OpenAI, DeepMind, etc.

Let's configure and train!

## Step 3: Configure PPO for Trading

In [ ]:
# Choose your contract
CONTRACT = 'MCL-1m'  # Options: 'MCL-1m', 'MGC-1m', 'mes-1m', 'ng', 'si'

# Define paths
DATA_DIR = Path('./data') / CONTRACT
WORK_DIR = Path('./saved_models') / f'{CONTRACT}_ppo'
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f"🎯 Contract: {CONTRACT}")
print(f"📁 Data: {DATA_DIR}")
print(f"💾 Models: {WORK_DIR}")

In [ ]:
# PPO Configuration
config = {
    'task_name': 'algorithmic_trading',
    'dataset_name': CONTRACT,
    'work_dir': str(WORK_DIR),
    
    # Data
    'data': {
        'type': 'AlgorithmicTradingDataset',
        'data_path': str(DATA_DIR),
        'train_path': str(DATA_DIR / 'train.csv'),
        'valid_path': str(DATA_DIR / 'valid.csv'),
        'test_path': str(DATA_DIR / 'test.csv'),
        'tech_indicator_list': [
            'high', 'low', 'open', 'close', 'adjcp',
            'zopen', 'zhigh', 'zlow', 'zadjcp', 'zclose',
            'zd_5', 'zd_10', 'zd_15', 'zd_20', 'zd_25', 'zd_30'
        ],
        'backward_num_day': 5,
        'forward_num_day': 5,
        'test_dynamic': '-1'
    },
    
    'environment': {
        'type': 'AlgorithmicTradingEnvironment'
    },
    
    # PPO Agent (different from DQN!)
    'agent': {
        'type': 'AlgorithmicTradingPPO',
        'max_step': 10000,
        'reward_scale': 1,
        'gamma': 0.99,  # Higher discount for PPO
        'lambda_gae_adv': 0.95,  # GAE parameter
        'lambda_entropy': 0.01,  # Entropy bonus for exploration
        'ratio_clip': 0.2,  # PPO clipping parameter (key!)
        'repeat_times': 4,  # Number of mini-batch updates
        'batch_size': 64,
    },
    
    # Training
    'trainer': {
        'type': 'AlgorithmicTradingTrainer',
        'epochs': 5,  # Start with 5
        'work_dir': str(WORK_DIR),
        'seeds_list': (42,),
        'batch_size': 64,
        'horizon_len': 256,  # PPO uses longer horizons
        'buffer_size': 100000,
        'num_threads': 4,
        'if_remove': False,
        'if_discrete': True,
        'if_off_policy': False,  # PPO is on-policy!
        'if_keep_save': True,
        'if_over_write': False,
        'if_save_buffer': False
    },
    
    'loss': {'type': 'MSELoss'},
    'optimizer': {'type': 'Adam', 'lr': 0.0003},  # Lower LR for PPO
    
    # Actor-Critic Network (PPO has both!)
    'act': {
        'type': 'ActorPPO',
        'state_dim': 82,
        'action_dim': 3,
        'dims': (128, 64),  # Larger network
    },
    
    'cri': {
        'type': 'CriticPPO',
        'state_dim': 82,
        'dims': (128, 64),
    },
    
    'transition': {'type': 'Transition'},
    'batch_size': 64
}

cfg = Config(config)
cfg = replace_cfg_vals(cfg)

print("✅ PPO Configuration created!")
print(f"\n🎯 Key PPO Parameters:")
print(f"  - Ratio clip: {cfg.agent['ratio_clip']} (prevents large policy updates)")
print(f"  - Lambda GAE: {cfg.agent['lambda_gae_adv']} (advantage estimation)")
print(f"  - Entropy: {cfg.agent['lambda_entropy']} (exploration bonus)")
print(f"  - Horizon: {cfg.trainer['horizon_len']} (longer than DQN)")

## Step 4: Build PPO Components

PPO needs both an **Actor** (policy) and a **Critic** (value function)!

In [ ]:
print("🔨 Building PPO components...\n")

# Dataset
print("📊 Dataset...")
dataset = build_dataset(cfg)
print(f"   ✅ {len(dataset.train_df):,} samples")

# Environments
print("\n🌍 Environments...")
train_env = build_environment(cfg, task='train')
valid_env = build_environment(cfg, task='valid')
print(f"   ✅ Created (state_dim={train_env.state_dim}, action_dim={train_env.action_dim})")

# Actor network (policy)
print("\n🎭 Actor (Policy Network)...")
act = build_net(cfg.act)
print(f"   ✅ {sum(p.numel() for p in act.parameters()):,} parameters")

# Critic network (value function)
print("\n🎯 Critic (Value Network)...")
cri = build_net(cfg.cri)
print(f"   ✅ {sum(p.numel() for p in cri.parameters()):,} parameters")

# Optimizers (separate for actor and critic)
print("\n⚙️ Optimizers...")
act_optimizer = build_optimizer(cfg, act)
cri_optimizer = build_optimizer(cfg, cri)
criterion = build_loss(cfg)
print(f"   ✅ Actor & Critic optimizers ready")

# Transition
print("\n💾 Transition buffer...")
transition = build_transition(cfg)
print(f"   ✅ Ready")

# PPO Agent
print("\n🤖 PPO Agent...")
agent = build_agent(cfg, train_env, act, cri, act_optimizer, cri_optimizer, criterion)
print(f"   ✅ Created with Actor-Critic architecture!")

# Trainer
print("\n🎓 Trainer...")
trainer = build_trainer(cfg, train_env, valid_env, agent, transition)
print(f"   ✅ Ready to train!")

print("\n" + "="*60)
print("✅ All PPO components ready!")
print("="*60)

## Step 5: Train PPO Agent 🚀

PPO training is more stable but may take slightly longer. The agent learns:
1. **Policy** (what actions to take)
2. **Value function** (how good is the current state)
3. Both are updated together for better stability!

In [ ]:
print("🚀 Training PPO agent...\n")
print("PPO is more stable but may take 15-40 minutes.")
print("The magic of clipped policy updates! ✨")
print("\n" + "="*60)

# Train
trainer.train_and_valid()

print("\n" + "="*60)
print("🎉 PPO Training complete!")
print("="*60)

## Step 6: Evaluate PPO Performance

In [ ]:
print("📈 Testing PPO agent...\n")

# Test
test_env = build_environment(cfg, task='test')
trainer.test()

print("\n✅ Evaluation complete!")

## Step 7: Visualize PPO Results

In [ ]:
# Load results
result_path = WORK_DIR / 'test' / f'{CONTRACT}_algorithmic_trading_test.csv'

if result_path.exists():
    results_df = pd.read_csv(result_path)
    
    fig, axes = plt.subplots(3, 1, figsize=(15, 12))
    
    # Portfolio Value
    axes[0].plot(results_df.index, results_df['portfolio_value'], linewidth=2, color='blue', label='PPO Agent')
    axes[0].axhline(y=test_env.initial_amount, color='red', linestyle='--', label='Initial Capital')
    axes[0].set_title('PPO: Portfolio Value Over Time', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Portfolio Value ($)', fontsize=12)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Actions
    if 'action' in results_df.columns:
        action_colors = {0: 'red', 1: 'gray', 2: 'green'}
        action_labels = {0: 'Sell', 1: 'Hold', 2: 'Buy'}
        for action, color in action_colors.items():
            mask = results_df['action'] == action
            axes[1].scatter(results_df.index[mask], [action]*mask.sum(), 
                          c=color, alpha=0.6, label=action_labels[action], s=10)
        axes[1].set_title('PPO: Trading Actions', fontsize=14, fontweight='bold')
        axes[1].set_ylabel('Action', fontsize=12)
        axes[1].set_yticks([0, 1, 2])
        axes[1].set_yticklabels(['Sell', 'Hold', 'Buy'])
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
    
    # Returns
    if 'portfolio_return' in results_df.columns:
        axes[2].plot(results_df.index, results_df['portfolio_return'], linewidth=1, alpha=0.7, color='blue')
        axes[2].axhline(y=0, color='red', linestyle='--')
        axes[2].set_title('PPO: Returns', fontsize=14, fontweight='bold')
        axes[2].set_xlabel('Time Step', fontsize=12)
        axes[2].set_ylabel('Return', fontsize=12)
        axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Statistics
    print("\n📊 PPO Performance Summary:\n")
    print("="*60)
    final_value = results_df['portfolio_value'].iloc[-1]
    initial_value = test_env.initial_amount
    total_return = (final_value - initial_value) / initial_value * 100
    
    print(f"Initial Capital:       ${initial_value:,.2f}")
    print(f"Final Portfolio Value: ${final_value:,.2f}")
    print(f"Total Return:          {total_return:+.2f}%")
    print(f"Profit/Loss:           ${final_value - initial_value:+,.2f}")
    
    if 'portfolio_return' in results_df.columns:
        returns = results_df['portfolio_return'].dropna()
        if len(returns) > 0:
            sharpe = returns.mean() / (returns.std() + 1e-8) * np.sqrt(252 * 1440)
            volatility = returns.std() * np.sqrt(252 * 1440)
            print(f"\nSharpe Ratio (Ann):    {sharpe:.2f}")
            print(f"Volatility (Ann):      {volatility:.2f}")
            print(f"Max Drawdown:          {(results_df['portfolio_value'].min() - initial_value) / initial_value * 100:.2f}%")
    
    print("="*60)
else:
    print(f"⚠️  Results not found at {result_path}")

## Step 8: Compare DQN vs PPO

If you've run both notebooks, let's compare the algorithms!

In [ ]:
# Try to load DQN results for comparison
dqn_path = Path('./saved_models') / CONTRACT / 'test' / f'{CONTRACT}_algorithmic_trading_test.csv'
ppo_path = result_path

if dqn_path.exists() and ppo_path.exists():
    dqn_df = pd.read_csv(dqn_path)
    ppo_df = pd.read_csv(ppo_path)
    
    fig, axes = plt.subplots(2, 1, figsize=(15, 10))
    
    # Portfolio comparison
    axes[0].plot(dqn_df.index, dqn_df['portfolio_value'], label='DQN', linewidth=2, color='green')
    axes[0].plot(ppo_df.index, ppo_df['portfolio_value'], label='PPO', linewidth=2, color='blue')
    axes[0].axhline(y=test_env.initial_amount, color='red', linestyle='--', label='Initial')
    axes[0].set_title('DQN vs PPO: Portfolio Value Comparison', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Portfolio Value ($)', fontsize=12)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Returns comparison
    if 'portfolio_return' in dqn_df.columns and 'portfolio_return' in ppo_df.columns:
        axes[1].plot(dqn_df.index, dqn_df['portfolio_return'].cumsum(), 
                    label='DQN Cumulative', linewidth=2, color='green', alpha=0.7)
        axes[1].plot(ppo_df.index, ppo_df['portfolio_return'].cumsum(), 
                    label='PPO Cumulative', linewidth=2, color='blue', alpha=0.7)
        axes[1].set_title('Cumulative Returns Comparison', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Time Step', fontsize=12)
        axes[1].set_ylabel('Cumulative Return', fontsize=12)
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Comparison table
    print("\n📊 Algorithm Comparison:\n")
    print("="*70)
    print(f"{'Metric':<25} {'DQN':>20} {'PPO':>20}")
    print("="*70)
    
    dqn_return = (dqn_df['portfolio_value'].iloc[-1] - test_env.initial_amount) / test_env.initial_amount * 100
    ppo_return = (ppo_df['portfolio_value'].iloc[-1] - test_env.initial_amount) / test_env.initial_amount * 100
    
    print(f"{'Total Return (%)':<25} {dqn_return:>19.2f}% {ppo_return:>19.2f}%")
    print(f"{'Final Value ($)':<25} {dqn_df['portfolio_value'].iloc[-1]:>20,.2f} {ppo_df['portfolio_value'].iloc[-1]:>20,.2f}")
    
    if 'portfolio_return' in dqn_df.columns:
        dqn_sharpe = dqn_df['portfolio_return'].mean() / (dqn_df['portfolio_return'].std() + 1e-8) * np.sqrt(252*1440)
        ppo_sharpe = ppo_df['portfolio_return'].mean() / (ppo_df['portfolio_return'].std() + 1e-8) * np.sqrt(252*1440)
        print(f"{'Sharpe Ratio':<25} {dqn_sharpe:>20.2f} {ppo_sharpe:>20.2f}")
    
    print("="*70)
    
    winner = 'DQN' if dqn_return > ppo_return else 'PPO'
    print(f"\n🏆 Winner: {winner} (on this dataset)")
    print("\n💡 Note: Results can vary with different seeds and hyperparameters!")
else:
    print("⚠️  Run Notebook 2 (DQN) first to compare algorithms!")

## Summary and Key Takeaways

### 🎉 Congratulations!

You've mastered PPO - one of the most powerful RL algorithms! Here's what you learned:

1. ✅ **PPO fundamentals** (policy-based, Actor-Critic)
2. ✅ **Why PPO is more stable** (clipped updates, entropy bonus)
3. ✅ **Training PPO** on intraday data
4. ✅ **Comparing with DQN** (different strengths)

### 🧠 Key Concepts:

**PPO Architecture:**
- **Actor**: Learns the policy (what to do)
- **Critic**: Estimates value (how good is it)
- **Clipping**: Prevents destructive updates (ratio_clip=0.2)
- **GAE**: Better advantage estimation
- **Entropy**: Encourages exploration

**PPO vs DQN:**
- PPO is generally **more stable**
- DQN can be **faster** to train
- PPO works with **continuous actions**
- DQN is simpler to understand

### 💡 When to use PPO:
- ✅ When you need **stable training**
- ✅ For **complex strategies**
- ✅ With **continuous action spaces**
- ✅ When **sample efficiency** matters

### 🔧 Experiment Ideas:
1. **Adjust clipping**: Try ratio_clip from 0.1 to 0.3
2. **Change entropy**: Higher = more exploration
3. **Longer horizons**: Increase horizon_len to 512
4. **Network size**: Larger networks for complex patterns

### 🚀 Next Steps:
In **Notebook 4**, you'll learn:
- **SAC (Soft Actor-Critic)** - Automatic exploration!
- Maximum entropy RL
- Continuous action spaces
- State-of-the-art performance

Ready for the most advanced algorithm? Let's go! 🎯

---
**Pro Tips**:
- PPO is great for **production systems** (stability)
- Combine with **risk management** (next notebook!)
- Try different **hyperparameters** for your data
- PPO often outperforms DQN with **more training** 💡